# 09 Neural networks

Feedforward networks `nn_1` to `nn_5` (one to five hidden layers of 32, 16, 8, 4 and 2 neurons) on the parsimonious predictor set of the `_mv25` samples, with the rolling windows of the other models. Settings follow Drobetz and Otto (2021): ReLU, batch normalisation, Adam, mini-batches of 1,000, at most 100 epochs, early stopping with a patience of 25, input dropout and an ensemble of ten seeds. The L1 penalty and the learning rate are tuned over the grids of Gu, Kelly and Xiu (2020), the dropout rate over {0, 0.25}. Forecasts are level-calibrated on the training sample (see `train_ensemble`) and joined with those of notebooks 07 and 08. Sections 3.6.4 and 3.7, Chapter 4.

Needs a GPU runtime: a little over four hours on a T4. Each architecture and year is saved as a checkpoint, so an interrupted run resumes where it stopped.

## Configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, time, gc, json, glob
import numpy as np
import pandas as pd
import torch

EST_DIR   = "/content/drive/MyDrive/Thesis/data"   # estimation samples (notebook 6)
PANEL_DIR = "/content/drive/MyDrive/Thesis/data"   # panel_characteristics.parquet (notebook 5)
PRED_DIR  = "/content/drive/MyDrive/Thesis/data"   # predictions_all_*.parquet (notebook 8)
OUT_DIR   = "/content/drive/MyDrive/Thesis/data"   # results of this notebook
CKPT_DIR  = os.path.join(OUT_DIR, "nn_checkpoints")

SAMPLES = ["complete_case_mv25", "imputed_mv25"]   # complete-case first, imputed second
FIRST_TEST, LAST_TEST = 2010, 2025
N_TRAIN, N_VAL = 8, 2
SEED = 0
NN_VERSION = "level-calibrated"   # stored with the checkpoints; a change forces re-estimation

NN_ARCHS = ["nn_1", "nn_2", "nn_3", "nn_4", "nn_5"]   # as in Drobetz and Otto; nn_1 to nn_3 only halves the time
L1_GRID, LR_GRID, DROPOUT_GRID = (1e-5, 1e-4, 1e-3), (0.001, 0.01), (0.0, 0.25)
N_SEEDS, BATCH, MAX_EPOCHS, PATIENCE = 10, 1000, 100, 25

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}" + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else
      "  -- WARNING: no GPU. Switch to a GPU runtime, or the estimation will take more than a day."))

## Functions

In [ ]:
import os, json, warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)

CHARS = ["size", "bm", "operatingprofitability", "totalassetgrowth", "ret_2_12",
         "issues_1_36", "accrualschange", "roa", "investment", "ret_1",
         "ret_12_36", "dy", "beta", "vola", "turnover", "debttoprice",
         "salestoprice", "cftoprice", "earningstoprice", "issues_1_12",
         "fcdispersion", "grossprofitability"]

PRIOR = ["ols3", "ols_pars", "ols_full", "elanet", "rf", "gbrt"]   # forecasts from notebooks 7 and 8

# Drobetz and Otto (2021), Table 2, test period 2000-2020
DO_R2 = {"ols_full": -0.52, "ols_pars": 0.57, "elanet": 1.21, "rf": 1.18, "gbrt": 1.14, "nn_1": 1.23}
DO_R2_LARGE = {"ols_full": -1.16, "ols_pars": 0.11, "elanet": 0.75, "rf": 0.92, "gbrt": 0.74, "nn_1": 0.93}
DO_R2_SMALL = {"ols_full": -0.10, "ols_pars": 0.90, "elanet": 1.55, "rf": 1.40, "gbrt": 1.46, "nn_1": 1.47}
DO_PS = {"ols_full": 0.45, "ols_pars": 0.65, "elanet": 0.92, "rf": 1.03, "gbrt": 1.07, "nn_1": 0.98}


# Data

def load_sample(name, est_dir, panel_dir):
    est = pd.read_parquet(os.path.join(est_dir, f"estimation_{name}.parquet"))
    est["date"] = pd.to_datetime(est["date"])
    raw = pd.read_parquet(os.path.join(panel_dir, "panel_characteristics.parquet"),
                          columns=["symbol", "date", "target"])
    raw["date"] = pd.to_datetime(raw["date"])
    return (est.merge(raw.rename(columns={"target": "raw_target"}), on=["symbol", "date"],
                      how="left").sort_values(["date", "symbol"]).reset_index(drop=True))


def build_parsimonious(df, chars=CHARS):
    """22 rank-transformed characteristics plus country and sector indicators
    (one category of each dropped), as for the linear `ols_pars` model."""
    dummies = pd.concat([
        pd.get_dummies(df["country"], prefix="cty", drop_first=True, dtype=np.float32),
        pd.get_dummies(df["sector"], prefix="sec", drop_first=True, dtype=np.float32)], axis=1)
    X = np.hstack([df[chars].to_numpy(dtype=np.float32), dummies.to_numpy()])
    return X, list(chars) + list(dummies.columns)


def rolling_splits(dates, first_test_year, last_test_year, n_train=8, n_val=2):
    years = pd.DatetimeIndex(dates).year
    for y in range(first_test_year, last_test_year + 1):
        tr = (years >= y - n_val - n_train) & (years <= y - n_val - 1)
        va = (years >= y - n_val) & (years <= y - 1)
        te = years == y
        if te.sum() and tr.sum() and va.sum():
            yield y, np.asarray(tr), np.asarray(va), np.asarray(te)

# Evaluation

def r2_oos(y, yhat):
    y, yhat = np.asarray(y), np.asarray(yhat)
    return 1.0 - np.sum((y - yhat) ** 2) / np.sum(y ** 2)


def r2_cross_section(df, m, y="target"):
    yy = df[y] - df.groupby("date")[y].transform("mean")
    ff = df[m] - df.groupby("date")[m].transform("mean")
    return 1.0 - ((yy - ff) ** 2).sum() / (yy ** 2).sum()


def predictive_slope(y, yhat):
    yhat = np.asarray(yhat)
    v = np.var(yhat)
    return np.cov(np.asarray(y), yhat, bias=True)[0, 1] / v if v > 0 else np.nan


def newey_west_se(x, lags=None):
    x = np.asarray(x, dtype=float)
    T = len(x)
    if lags is None:
        lags = int(np.floor(4 * (T / 100) ** (2 / 9)))
    u = x - x.mean()
    s = np.mean(u ** 2)
    for l in range(1, lags + 1):
        s += 2 * (1 - l / (lags + 1)) * np.mean(u[l:] * u[:-l])
    return np.sqrt(s / T)


def diebold_mariano(df, model_a, model_b, y="target"):
    """Positive: model_b more accurate than model_a."""
    d = ((df[y] - df[model_a]) ** 2 - (df[y] - df[model_b]) ** 2).groupby(df["date"]).mean()
    return d.mean() / newey_west_se(d.values)


def decile_portfolios(df, pred, ret="raw_target", weight="MV"):
    d = df.dropna(subset=[pred, ret]).copy()
    d["dec"] = d.groupby("date")[pred].transform(
        lambda s: pd.qcut(s.rank(method="first"), 10, labels=False))
    ew = d.groupby(["date", "dec"])[ret].mean().unstack()
    d["_w"] = d[weight].clip(lower=0)
    d["_wr"] = d["_w"] * d[ret]
    g = d.groupby(["date", "dec"])
    vw = (g["_wr"].sum(min_count=1) / g["_w"].sum(min_count=1)).unstack()
    return ew, vw


def portfolio_stats(port):
    hl = (port[9] - port[0]).dropna()
    return {"mean_%": 100 * hl.mean(), "t_stat": hl.mean() / newey_west_se(hl.values),
            "sharpe_ann": hl.mean() / hl.std() * np.sqrt(12), "months": len(hl)}


def annual_r2(P, models, y="target"):
    return pd.DataFrame({m: P.groupby(P["date"].dt.year).apply(
        lambda g: 100 * r2_oos(g[y], g[m]), include_groups=False) for m in models})


def periods_of(P):
    yr = P["date"].dt.year
    return {"2010-2025": yr >= 2010, "2010-2020": yr <= 2020, "2021-2025": yr >= 2021}


def avg_annual_table(P, models, y="target"):
    a = annual_r2(P, models, y)
    return pd.DataFrame({"2010-2025": a.mean(), "2010-2020": a.loc[:2020].mean(),
                         "2021-2025": a.loc[2021:].mean()})


def side_by_side(tables, ref=None, ref_label="DO"):
    out = pd.concat(tables, axis=1)
    if ref is not None:
        out[(ref_label, "2000-2020")] = pd.Series(ref)
    return out




In [ ]:
import numpy as np
import torch

ARCHS = {"nn_1": [32], "nn_2": [32, 16], "nn_3": [32, 16, 8],
         "nn_4": [32, 16, 8, 4], "nn_5": [32, 16, 8, 4, 2]}


class StackedMLP:
    """
    M independent networks with the same architecture, stored as stacked weight
    tensors and trained in a single pass. No parameter is shared and the loss is the
    sum of the members' losses, so this is equivalent to training them one at a time.
    Dropout acts on the inputs (one rate per member); hidden layers are
    linear -> batch normalisation -> ReLU, the output layer is linear.
    """

    def __init__(self, n_in, hidden, M, dropout, device, gen):
        self.hidden, self.M, self.device = hidden, M, device
        d = torch.as_tensor(dropout, dtype=torch.float32, device=device)
        self.dropout = d.expand(M).clone() if d.dim() == 0 else d
        self.params, self.buffers = {}, {}
        sizes = [n_in] + hidden
        for k in range(len(hidden)):
            fi, fo = sizes[k], sizes[k + 1]
            self.params[f"W{k}"] = (torch.randn(M, fi, fo, generator=gen, device=device)
                                    * np.sqrt(2.0 / fi)).requires_grad_()            # He initialisation
            self.params[f"b{k}"] = torch.zeros(M, 1, fo, device=device, requires_grad=True)
            self.params[f"g{k}"] = torch.ones(M, 1, fo, device=device, requires_grad=True)
            self.params[f"be{k}"] = torch.zeros(M, 1, fo, device=device, requires_grad=True)
            self.buffers[f"rm{k}"] = torch.zeros(M, 1, fo, device=device)
            self.buffers[f"rv{k}"] = torch.ones(M, 1, fo, device=device)
        fi = sizes[-1]
        self.params["Wout"] = (torch.randn(M, fi, 1, generator=gen, device=device)
                               * np.sqrt(1.0 / fi)).requires_grad_()
        self.params["bout"] = torch.zeros(M, 1, 1, device=device, requires_grad=True)
        self.weight_keys = [k for k in self.params if k.startswith("W")]

    def forward(self, x, train, gen=None):
        """x: (M, B, n_in) in training, or (B, n_in) shared by all members in evaluation."""
        h = x
        if train and bool((self.dropout > 0).any()):
            rate = self.dropout.view(-1, 1, 1)
            keep = (torch.rand(h.shape, generator=gen, device=self.device) >= rate).float()
            h = h * keep / (1.0 - rate)
        for k in range(len(self.hidden)):
            W, b = self.params[f"W{k}"], self.params[f"b{k}"]
            z = (torch.einsum("bp,mph->mbh", h, W) if h.dim() == 2 else torch.bmm(h, W)) + b
            if train:
                mu = z.mean(dim=1, keepdim=True)
                var = z.var(dim=1, unbiased=False, keepdim=True)
                with torch.no_grad():
                    self.buffers[f"rm{k}"].mul_(0.9).add_(0.1 * mu)
                    self.buffers[f"rv{k}"].mul_(0.9).add_(0.1 * var)
            else:
                mu, var = self.buffers[f"rm{k}"], self.buffers[f"rv{k}"]
            z = (z - mu) / torch.sqrt(var + 1e-5) * self.params[f"g{k}"] + self.params[f"be{k}"]
            h = torch.relu(z)
        W = self.params["Wout"]
        out = (torch.einsum("bp,mph->mbh", h, W) if h.dim() == 2 else torch.bmm(h, W)) + self.params["bout"]
        return out.squeeze(-1)

    def l1(self):
        """L1 norm of the weights of each member (biases and normalisation parameters excluded)."""
        return sum(self.params[k].abs().sum(dim=(1, 2)) for k in self.weight_keys)

    def state(self):
        return {k: v.detach().clone() for k, v in {**self.params, **self.buffers}.items()}

    def load(self, st):
        for k, v in st.items():
            (self.params if k in self.params else self.buffers)[k].data.copy_(v)

    @torch.no_grad()
    def predict(self, X, chunk=8192):
        return torch.cat([self.forward(X[i:i + chunk], train=False)
                          for i in range(0, X.shape[0], chunk)], dim=1)   # (M, N)


def train_ensemble(Xtr, ytr, Xva, yva, hidden, l1_grid, lr_grid, dropout_grid, n_seeds,
                   batch_size, max_epochs, patience, seed, device):
    """
    Train all (L1 penalty, learning rate, dropout, seed) combinations of one
    architecture together, with Adam and early stopping on the validation sample.
    At each evaluation the forecasts are shifted so that their average over the
    training sample equals the average training return; otherwise early stopping
    tends to pick the epoch whose output bias matches the average return of the
    validation years.
    """
    combos = [(l1, lr, d) for l1 in l1_grid for lr in lr_grid for d in dropout_grid]
    M = len(combos) * n_seeds
    rep = lambda j: torch.tensor([c[j] for c in combos for _ in range(n_seeds)],
                                 dtype=torch.float32, device=device)
    lam, lrs, drop = rep(0), rep(1), rep(2)
    gen = torch.Generator(device=device).manual_seed(seed)
    net = StackedMLP(Xtr.shape[1], hidden, M, drop, device, gen)

    # Adam with a learning rate per member (Kingma and Ba, 2014)
    b1, b2, eps = 0.9, 0.999, 1e-8
    m1 = {k: torch.zeros_like(v) for k, v in net.params.items()}
    m2 = {k: torch.zeros_like(v) for k, v in net.params.items()}
    step = 0

    N = Xtr.shape[0]
    n_batches = max(1, N // batch_size)
    # training subsample for the level calibration
    sub = torch.randperm(N, generator=gen, device=device)[:min(N, 20_000)]
    ybar = ytr.mean()
    best_mse = torch.full((M,), float("inf"), device=device)
    best_epoch = torch.zeros(M, dtype=torch.long, device=device)
    best_state = net.state()
    active = torch.ones(M, dtype=torch.bool, device=device)

    for epoch in range(max_epochs):
        perm = torch.argsort(torch.rand(M, N, generator=gen, device=device), dim=1)
        for b in range(n_batches):
            idx = perm[:, b * batch_size:(b + 1) * batch_size]                 # (M, B)
            xb, yb = Xtr[idx], ytr[idx]
            pred = net.forward(xb, train=True, gen=gen)
            loss_m = ((yb - pred) ** 2).mean(dim=1) + lam * net.l1()
            for v in net.params.values():
                v.grad = None
            loss_m.sum().backward()
            step += 1
            with torch.no_grad():
                for k, p in net.params.items():
                    g = p.grad
                    m1[k].mul_(b1).add_((1 - b1) * g)
                    m2[k].mul_(b2).add_((1 - b2) * g * g)
                    mh = m1[k] / (1 - b1 ** step)
                    vh = m2[k] / (1 - b2 ** step)
                    upd = mh / (torch.sqrt(vh) + eps)
                    shape = (M,) + (1,) * (p.dim() - 1)
                    p.sub_(upd * (lrs * active.float()).view(shape))           # stopped members are frozen
        shift = ybar - net.predict(Xtr[sub]).mean(dim=1, keepdim=True)    # level calibration
        mse = ((yva.unsqueeze(0) - net.predict(Xva) - shift) ** 2).mean(dim=1)
        improved = (mse < best_mse) & active
        if improved.any():
            st = net.state()
            for k in st:
                best_state[k][improved] = st[k][improved]
            best_mse = torch.where(improved, mse, best_mse)
            best_epoch = torch.where(improved, torch.full_like(best_epoch, epoch), best_epoch)
        active = active & ((epoch - best_epoch) < patience)
        if not active.any():
            break
    net.load(best_state)
    labels = [c for c in combos for _ in range(n_seeds)]
    return net, labels, best_epoch.cpu().numpy() + 1, epoch + 1


def level_shift(net, Xtr, ytr):
    """Constant per member that sets the average training forecast equal to the average training return."""
    return (ytr.mean() - net.predict(Xtr).mean(dim=1, keepdim=True)).cpu().numpy()


def select_and_forecast(net, labels, Xva, yva, Xte, n_seeds, Xtr=None, ytr=None):
    """
    Average the seeds of each hyperparameter combination and return the test forecast
    of the combination with the lowest validation error. If the training sample is
    passed, the forecasts are first level-calibrated on it.
    """
    pv = net.predict(Xva).cpu().numpy()
    pt = net.predict(Xte).cpu().numpy()
    if Xtr is not None:                              # level calibrated on the training sample
        sh = level_shift(net, Xtr, ytr)
        pv, pt = pv + sh, pt + sh
    yv = yva.cpu().numpy()
    best = None
    for g in range(len(labels) // n_seeds):
        sl = slice(g * n_seeds, (g + 1) * n_seeds)
        mse = float(np.mean((yv - pv[sl].mean(axis=0)) ** 2))
        if best is None or mse < best["val_mse"]:
            best = {"val_mse": mse, "combo": labels[g * n_seeds], "members": list(range(sl.start, sl.stop)),
                    "forecast": pt[sl].mean(axis=0)}
    return best


## Estimation

In [ ]:
t0 = time.time()
for s in SAMPLES:
    ck = os.path.join(CKPT_DIR, s)
    os.makedirs(ck, exist_ok=True)
    est = load_sample(s, EST_DIR, PANEL_DIR)
    X, names = build_parsimonious(est)
    y = est["target"].to_numpy(dtype=np.float32)

    fp = NN_VERSION + ":" + str(int(pd.util.hash_pandas_object(est[["date", "target"] + CHARS].iloc[::101],
                                                              index=False).sum()))
    f_fp = os.path.join(ck, "fingerprint.txt")
    has_preds = len(glob.glob(os.path.join(ck, "*.parquet"))) > 0
    if (os.path.exists(f_fp) and open(f_fp).read().strip() != fp) or (has_preds and not os.path.exists(f_fp)):
        raise RuntimeError(f"The checkpoints in {ck} were not produced from the current version of the "
                           f"{s} sample or of the estimation code. Delete the folder {CKPT_DIR} and rerun this cell.")
    with open(f_fp, "w") as fh:
        fh.write(fp)
    print(f"\n{s}: {len(est):,} security-months, {X.shape[1]} predictors")

    for yr, tr, va, te in rolling_splits(est["date"], FIRST_TEST, LAST_TEST, N_TRAIN, N_VAL):
        todo = [a for a in NN_ARCHS if not os.path.exists(os.path.join(ck, f"{yr}_{a}.parquet"))]
        if not todo:
            print(f"   {yr}: already estimated, skipped")
            continue
        Xtr = torch.tensor(X[tr], device=DEVICE); ytr = torch.tensor(y[tr], device=DEVICE)
        Xva = torch.tensor(X[va], device=DEVICE); yva = torch.tensor(y[va], device=DEVICE)
        Xte = torch.tensor(X[te], device=DEVICE)
        msg = []
        for a in todo:
            t1 = time.time()
            seed = SEED + 1000 * yr + NN_ARCHS.index(a)
            net, labels, best_ep, n_ep = train_ensemble(Xtr, ytr, Xva, yva, ARCHS[a], L1_GRID, LR_GRID,
                                                        DROPOUT_GRID, N_SEEDS, BATCH, MAX_EPOCHS, PATIENCE,
                                                        seed, DEVICE)
            best = select_and_forecast(net, labels, Xva, yva, Xte, N_SEEDS, Xtr, ytr)
            l1, lr, dr = best["combo"]
            with open(os.path.join(ck, f"{yr}_{a}.json"), "w") as fh:
                json.dump({"year": yr, "arch": a, "l1": l1, "lr": lr, "dropout": dr,
                           "mean_best_epoch": float(np.mean(best_ep[best["members"]])),
                           "epochs_run": int(n_ep)}, fh)
            out = est.loc[te, ["symbol", "date"]].copy()
            out[a] = best["forecast"]
            out.to_parquet(os.path.join(ck, f"{yr}_{a}.parquet"), index=False)   # written last
            msg.append(f"{a} l1 {l1:.0e} lr {lr} drop {dr} ({time.time()-t1:.0f}s)")
            del net; gc.collect()
            if DEVICE == "cuda":
                torch.cuda.empty_cache()
        print(f"   {yr}: " + " | ".join(msg), flush=True)
    del est, X, y; gc.collect()
print(f"\nDone in {(time.time()-t0)/60:.1f} min.")

## Assembling the forecasts

In [ ]:
MODELS = PRIOR + NN_ARCHS
R = {}
for s in SAMPLES:
    ck = os.path.join(CKPT_DIR, s)
    P = pd.read_parquet(os.path.join(PRED_DIR, f"predictions_all_{s}.parquet"))
    P["date"] = pd.to_datetime(P["date"])
    for a in NN_ARCHS:
        parts = [pd.read_parquet(f) for f in sorted(glob.glob(os.path.join(ck, f"*_{a}.parquet")))]
        nn = pd.concat(parts, ignore_index=True)
        nn["date"] = pd.to_datetime(nn["date"])
        P = P.merge(nn, on=["symbol", "date"], how="left")
    T = pd.DataFrame([json.load(open(f)) for f in sorted(glob.glob(os.path.join(ck, "*.json")))])
    R[s] = {"P": P, "T": T}
    miss = P[NN_ARCHS].isna().sum().to_dict()
    print(f"{s:20s} {len(P):>9,} forecasts | network forecasts missing: {miss}")

## Each sample on its own observations

Drobetz and Otto report 1.23% for nn_1, their best network.

In [ ]:
avg = side_by_side({s: avg_annual_table(R[s]["P"], MODELS) for s in SAMPLES}, DO_R2)
print("Average of annual R^2 (%):")
print(avg.round(3).to_string())

pooled = pd.concat({s: pd.DataFrame({p: {m: 100 * r2_oos(R[s]["P"].loc[mk, "target"], R[s]["P"].loc[mk, m])
                                         for m in MODELS} for p, mk in periods_of(R[s]["P"]).items()})
                    for s in SAMPLES}, axis=1)
print("\nPooled R^2 (%):")
print(pooled.round(3).to_string())

In [ ]:
for s in SAMPLES:
    print(f"\nR^2 (%) by test year -- {s}:")
    print(annual_r2(R[s]["P"], MODELS).round(2).to_string())

### Ranking stocks versus forecasting the market

In [ ]:
cs = pd.concat({s: pd.DataFrame({p: {m: 100 * r2_cross_section(R[s]["P"][mk], m) for m in MODELS}
                                  for p, mk in periods_of(R[s]["P"]).items()}) for s in SAMPLES}, axis=1)
print("Cross-sectional R^2 (%):")
print(cs.round(3).to_string())

corr = {}
for s in SAMPLES:
    P = R[s]["P"]
    mkt = P.groupby("date")["target"].mean().groupby(lambda d: d.year).mean()
    corr[s] = annual_r2(P, MODELS).corrwith(mkt)
print("\nCorrelation of annual R^2 with the year's average excess return:")
print(pd.DataFrame(corr).round(2).to_string())

### Large and small firms

In [ ]:
size_tabs = {}
for s in SAMPLES:
    P = R[s]["P"]
    large = P["MV"] >= P.groupby("date")["MV"].transform("median")
    size_tabs[s] = pd.DataFrame({"large": annual_r2(P[large], MODELS).mean(),
                                 "small": annual_r2(P[~large], MODELS).mean()})
size_r2 = pd.concat(size_tabs, axis=1)
size_r2[("DO", "large")] = pd.Series(DO_R2_LARGE)
size_r2[("DO", "small")] = pd.Series(DO_R2_SMALL)
print("Average of annual R^2 (%) by size, 2010-2025:")
print(size_r2.round(3).to_string())

### Predictive slopes

Drobetz and Otto report 0.98 for nn_1.

In [ ]:
ps = {}
for s in SAMPLES:
    P = R[s]["P"]
    a = pd.DataFrame({m: P.groupby(P["date"].dt.year).apply(
        lambda g: predictive_slope(g["target"], g[m]), include_groups=False) for m in MODELS})
    ps[s] = pd.DataFrame({"2010-2025": a.mean(), "2010-2020": a.loc[:2020].mean(),
                          "2021-2025": a.loc[2021:].mean()})
    unstable = a.stack()[a.stack().abs() > 5]
    if len(unstable):
        print(f"{s}: unstable slopes (|PS| > 5): "
              + ", ".join(f"{m} {y}: {v:.1f}" for (y, m), v in unstable.items()))
ps_tab = side_by_side(ps, DO_PS)
print(ps_tab.round(3).to_string())

### Diebold-Mariano tests

Positive: the column model is more accurate than the row model; |DM| > 1.96 is
significant at 5%.

In [ ]:
dm = {}
for s in SAMPLES:
    d = pd.DataFrame(index=MODELS, columns=MODELS, dtype=float)
    for a in MODELS:
        for b in MODELS:
            if a != b:
                d.loc[a, b] = diebold_mariano(R[s]["P"], a, b)
    dm[s] = d
    print(f"\n{s}:")
    print(d.round(2).to_string())

## Comparing the samples

### On the same observations

Positive DM: the model estimated on the imputed sample is more accurate.

In [ ]:
CMP = ["ols3", "elanet", "rf", "gbrt"] + NN_ARCHS
Pc, Pi = R[SAMPLES[0]]["P"], R[SAMPLES[1]]["P"]
common = Pc[["symbol", "date", "target"] + CMP].merge(
    Pi[["symbol", "date"] + CMP].rename(columns={m: f"{m}_imp" for m in CMP}),
    on=["symbol", "date"], how="inner")
print(f"Common observations: {len(common):,} of {len(Pc):,} complete-case forecasts\n")

rows = []
yr = common["date"].dt.year
for m in CMP:
    rows.append({"model": m,
                 "R2 trained CC": annual_r2(common, [m]).iloc[:, 0].mean(),
                 "R2 trained imputed": annual_r2(common.rename(columns={f"{m}_imp": "x"}), ["x"]).iloc[:, 0].mean(),
                 "CS R2 trained CC": 100 * r2_cross_section(common, m),
                 "CS R2 trained imputed": 100 * r2_cross_section(common, f"{m}_imp"),
                 "DM 2010-2025": diebold_mariano(common, m, f"{m}_imp"),
                 "DM 2010-2020": diebold_mariano(common[yr <= 2020], m, f"{m}_imp"),
                 "DM 2021-2025": diebold_mariano(common[yr >= 2021], m, f"{m}_imp")})
same_obs = pd.DataFrame(rows).set_index("model")
print(same_obs.round(3).to_string())

### On the recovered observations

In [ ]:
key_c = pd.MultiIndex.from_frame(Pc[["symbol", "date"]])
key_i = pd.MultiIndex.from_frame(Pi[["symbol", "date"]])
recovered, retained = Pi[~key_i.isin(key_c)], Pi[key_i.isin(key_c)]
print(f"Recovered: {len(recovered):,} forecasts ({100*len(recovered)/len(Pi):.1f}%), "
      f"median market value {recovered['MV'].median():.1f} mn against {retained['MV'].median():.1f} mn retained\n")
rec = pd.DataFrame({
    "R2 recovered": annual_r2(recovered, MODELS).mean(),
    "R2 retained": annual_r2(retained, MODELS).mean(),
    "CS R2 recovered": pd.Series({m: 100 * r2_cross_section(recovered, m) for m in MODELS}),
    "CS R2 retained": pd.Series({m: 100 * r2_cross_section(retained, m) for m in MODELS})})
print("Models estimated on the imputed sample, 2010-2025 (%):")
print(rec.round(3).to_string())

## Selected hyperparameters

L1 penalty, learning rate, dropout rate and average best epoch of the ten selected networks, by test year.

In [ ]:
for s in SAMPLES:
    T = R[s]["T"]
    print(f"\n{s}:")
    print(T.pivot(index="year", columns="arch", values="l1").to_string())
    print(T.pivot(index="year", columns="arch", values="lr").to_string())
    print(T.pivot(index="year", columns="arch", values="dropout").to_string())
    print(T.pivot(index="year", columns="arch", values="mean_best_epoch").round(1).to_string())

## Portfolio performance

Monthly deciles on the forecast, long-short top minus bottom, unwinsorised returns of
the following month.

In [ ]:
rows = []
for s in SAMPLES:
    for m in MODELS:
        ew, vw = decile_portfolios(R[s]["P"], m, ret="raw_target", weight="MV")
        for lab, port in [("equal", ew), ("value", vw)]:
            rows.append({"sample": s, "model": m, "weighting": lab, **portfolio_stats(port)})
perf = pd.DataFrame(rows)
print(perf.pivot_table(index="model", columns=["sample", "weighting"],
                       values=["mean_%", "t_stat", "sharpe_ann"]).round(2).reindex(MODELS).to_string())

for s in SAMPLES:
    ew, vw = decile_portfolios(R[s]["P"], "nn_1", ret="raw_target", weight="MV")
    print(f"\n{s} -- nn_1, mean monthly excess return by decile (%):")
    print(pd.DataFrame({"equal": 100 * ew.mean(), "value": 100 * vw.mean()}).round(3).T.to_string())

## Save

In [ ]:
TAG = SAMPLES[0].replace("complete_case", "")
for s in SAMPLES:
    R[s]["P"].to_parquet(os.path.join(OUT_DIR, f"predictions_final_{s}.parquet"), index=False)
    R[s]["T"].to_csv(os.path.join(OUT_DIR, f"tuning_nn_{s}.csv"), index=False)
for name, df in [("r2_avg_annual", avg), ("r2_pooled", pooled), ("r2_cross_section", cs),
                 ("r2_by_size", size_r2), ("predictive_slopes", ps_tab),
                 ("same_observations", same_obs), ("recovered", rec)]:
    df.to_csv(os.path.join(OUT_DIR, f"{name}_nn{TAG}.csv"))
perf.to_csv(os.path.join(OUT_DIR, f"portfolios_nn{TAG}.csv"), index=False)
for s in SAMPLES:
    dm[s].to_csv(os.path.join(OUT_DIR, f"diebold_mariano_nn_{s}.csv"))
print(f"Saved forecasts of all models, tuning and result tables (suffix '{TAG}').")

## Level of the forecasts
Average forecast and average return by test year. After the level calibration, the network forecasts follow the training years, as those of the elastic net and the random forest do.

In [ ]:
for s in SAMPLES:
    P = R[s]["P"]
    est = load_sample(s, EST_DIR, PANEL_DIR)
    ye = est["date"].dt.year
    rows = []
    for y in range(FIRST_TEST, LAST_TEST + 1):
        g = P[P["date"].dt.year == y]
        rows.append({"year": y,
                     "return, test": 100 * g["target"].mean(),
                     "return, training": 100 * est.loc[(ye >= y - N_VAL - N_TRAIN) & (ye < y - N_VAL), "target"].mean(),
                     "return, validation": 100 * est.loc[(ye >= y - N_VAL) & (ye < y), "target"].mean(),
                     **{m: 100 * g[m].mean() for m in ["elanet", "rf"] + NN_ARCHS}})
    d = pd.DataFrame(rows).set_index("year")
    print(f"\n{s}: average forecast and average return by test year (% per month)")
    print(d.round(2).to_string())
    print("\nCorrelation of each model's average forecast with the average return of the")
    print(f"{'':10s} {'validation years':>18s} {'training years':>16s}")
    for m in ["elanet", "rf"] + NN_ARCHS:
        print(f"  {m:8s} {d[m].corr(d['return, validation']):>18.2f} {d[m].corr(d['return, training']):>16.2f}")